# Importing libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer
nltk.download("stopwords")

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

# Importing dataset

In [ ]:
dataset = pd.read_csv('training.1600000.processed.noemoticon.csv', encoding='ISO-8859-1',header=None,engine='python')
dataset= dataset.groupby(0).sample(n=50000, random_state=0).reset_index(drop=True)
print(dataset[0].value_counts())
x=dataset.iloc[:,5].values
y=dataset.iloc[:,0].values
dataset.head()

0
0    50000
4    50000
Name: count, dtype: int64


,0,1,2,3,4,5
0,0,1956549353,Thu May 28 22:07:18 PDT 2009,NO_QUERY,binka629,"oh god whits pissed n shes drivin, in the rain..."
1,0,2179120188,Mon Jun 15 08:40:19 PDT 2009,NO_QUERY,xochrystalo7,driving back from myrtle beach
2,0,1985150739,Sun May 31 16:32:04 PDT 2009,NO_QUERY,MedoraChevalier,Saying byebye to AM's Refuge and Extension wi...
3,0,1557029956,Sun Apr 19 01:37:16 PDT 2009,NO_QUERY,Chisstwitt,@SarahSaner Did you she fell of her horse agai...
4,0,2033945132,Thu Jun 04 13:24:19 PDT 2009,NO_QUERY,xLiLShanx,@SalioElSol08 save me any.


# Spliting the test set and training set

In [ ]:
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test = train_test_split(x,y,test_size=0.2,random_state=42)
print(x_train[0:5])
print(x_test[0:5])
print(y_train[0:5])
print(y_test[0:5])

["@MrBigDreams yea ok both y'all crazy then!! Lol so ubertwitter the shit right??  lol"
 'is doing homework '
 'And for the record i can never watch &quot;Twilight&quot; again '
 'The Sun is up !!!!! and i have to study   but its okay !!!'
 'Why HELLO, Sun.  Fancy seeing you here.    You make my day a little brighter.']
["is recovering from what was the best night I've had out East...Gotta love NYC! "
 'God is Good, Life is Good ' 'english summative '
 '@dannycasler have a safe trip, Danny. Hope you enjoyed Ukraine '
 '@promisepromise ok bbz, i hope it works  love you lots xxx']
[4 0 0 0 4]
[4 4 0 4 4]


# Text cleaning

In [ ]:
corpus = []
all_stopwords = stopwords.words("english")
all_stopwords.remove('not')
all_stopwords.remove("isn't")
all_stopwords.remove("don't")
all_stopwords.remove("haven't")
all_stopwords.remove("couldn't")
all_stopwords.remove("wouldn't")
ps = PorterStemmer()
for n in range(len(x_train)):

    tweet = re.sub('[^a-zA-Z]',' ',str(x_train[n]))
    tweet = tweet.lower()
    tweet = tweet.split()

    tweet = [ps.stem(word) for word in tweet if not word in set(all_stopwords)]
    tweet =" ".join(tweet)
    corpus.append(tweet)

In [ ]:
corpus_test = []

for n in range(len(x_test)):

    tweet = re.sub('[^a-zA-Z]',' ',str(x_test[n]))
    tweet = tweet.lower()
    tweet = tweet.split()

    tweet = [ps.stem(word) for word in tweet if not word in set(all_stopwords)]
    tweet =" ".join(tweet)
    corpus_test.append(tweet)

In [ ]:
print(corpus[0:5])

['mrbigdream yea ok crazi lol ubertwitt shit right lol', 'homework', 'record never watch quot twilight quot', 'sun studi okay', 'hello sun fanci see make day littl brighter']


# Bag of words

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
cv = CountVectorizer(max_features=5000)
x_train=cv.fit_transform(corpus).toarray()
x_test=cv.transform(corpus_test).toarray()

# Isprobavanje logistic regression modela

In [ ]:
from sklearn.metrics import confusion_matrix, accuracy_score
from sklearn.linear_model import LogisticRegression
classifier_logistic = LogisticRegression(random_state = 0,max_iter=1000)
classifier_logistic.fit(x_train,y_train)
y_pred =classifier_logistic.predict(x_test)
from sklearn.metrics import confusion_matrix,accuracy_score
cm=confusion_matrix(y_test,y_pred)
print(cm)
ac=accuracy_score(y_test,y_pred)
print(ac)

[[7390 2645]
 [2193 7772]]
0.7581


# pickle

In [ ]:
import pickle
pickle.dump(cv,open('vectorizer.pkl','wb'))
pickle.dump(classifier_logistic,open('model.pkl','wb'))